# Data from an API and from a database

Python stage, lesson P23. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** send a request with parameters and read the status code; turn a JSON answer into a DataFrame; read a SQL query into pandas from a SQLite database; pass values to a query as parameters, never pasted into its text.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sqlite3
import sys
import traceback

import numpy as np
import pandas as pd
import requests

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)


def check_function(label, name, cases):
    """Call your function on every case. Each case is (arguments, keyword arguments, expected result)."""
    fn = globals().get(name)
    if not callable(fn):
        RESULTS[label] = None
        print(f"{label}: define a function called {name}, then run this cell again.")
        return False
    for args, kwargs, expected in cases:
        try:
            got = fn(*args, **kwargs)
        except Exception as error:
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} stopped with {type(error).__name__}: {error}")
            return False
        if not same(got, expected):
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} should give {expected!r}, but gave {got!r}.")
            if got is None:
                print("Hint: None means the function gave nothing back. Did it return the value, or only print it?")
            return False
    RESULTS[label] = True
    print(f"{label}: correct ({len(cases)} cases).")
    return True

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: Request, response, status code

**Predict first.** what will the status code be, and what will the address that was really sent look like?

In [ ]:
URL = "https://api.frankfurter.dev/v2/rates"
params = {"providers": "ecb", "from": "2026-01-05", "to": "2026-01-09", "base": "USD", "quotes": "EUR"}
response = requests.get(URL, params=params, timeout=30)
print(response.status_code)
print(response.url)

**Predict first.** will `requests.get` stop with an error for a page that is not there? What will the status code, `ok` and the text of the answer be?

In [ ]:
missing = requests.get("https://api.frankfurter.dev/v2/nothing", timeout=30)
print(missing.status_code, missing.ok)
print(missing.text)
show_error("missing.raise_for_status()", short=True)

## Unit 2: JSON in, DataFrame out

**Predict first.** how many records will there be, and on which day will the 5.00-dollar coffee be cheapest in euros?

In [ ]:
records = response.json()
print(type(records).__name__, len(records))
print(records[0])
rates = pd.DataFrame(records)
rates["date"] = pd.to_datetime(rates["date"])
rates["coffee_eur"] = (5.00 * rates["rate"]).round(4)
print(rates[["date", "rate", "coffee_eur"]])

## Unit 3: read_sql: a database as a source

**Predict first.** what will `per_store` hold: how many rows and columns, and which counts?

In [ ]:
con = sqlite3.connect(":memory:")
for name, table in {"stores": stores, "customers": customers, "products": products, "orders": orders, "order_items": order_items}.items():
    table.to_sql(name, con, index=False)
query = "SELECT store_id, COUNT(*) AS orders FROM orders GROUP BY store_id ORDER BY store_id"
per_store = pd.read_sql(query, con)
print(per_store)

## Unit 4: Parameters, never pasted text

**Predict first.** how many orders will store 3 have? Then the user typed `3 OR 1=1`: how many will the pasted query and the parameter query count?

In [ ]:
sql = "SELECT COUNT(*) AS orders FROM orders WHERE store_id = ?"
print(pd.read_sql(sql, con, params=(3,)).iloc[0, 0])
store = "3 OR 1=1"
pasted = pd.read_sql(f"SELECT COUNT(*) AS orders FROM orders WHERE store_id = {store}", con).iloc[0, 0]
bound = pd.read_sql(sql, con, params=(store,)).iloc[0, 0]
print(pasted, bound)

## Unit 5: Check: two routes to the same counts

**Predict first.** does the database give the same counts as `groupby` on the table, and what do they add up to?

In [ ]:
by_groupby = orders.groupby("store_id").size()
print(per_store["orders"].tolist() == by_groupby.tolist(), per_store["orders"].sum())

## Finish it

### Exercise 1: Finish it

**Task:** What was the average dollar to euro rate in the week of February 2 to 6, 2026? `week` holds the answer of the API for two currencies, euro and pound, one row for each currency and day. Replace the first `None` with only the euro rows, then set `answer_1` to their average rate, a decimal number.

**Expected:** one decimal number, below 1.

In [ ]:
week = pd.DataFrame(requests.get(URL, params={"providers": "ecb", "from": "2026-02-02", "to": "2026-02-06", "base": "USD", "quotes": "EUR,GBP"}, timeout=30).json())  # given
euro = None  # replace None: only the rows of the euro
answer_1 = None  # then set answer_1 to the average of the rate column of euro

In [ ]:
_ = check("Exercise 1", "answer_1", 0.846698, traps={0.78969: "That average mixes the euro and the pound rows. Keep only the rows whose quote is EUR."})

<details>
<summary>Hint</summary>

Print `week`. How many rows are there for each day? A boolean mask, as in lesson P12, keeps the rows of one currency.

</details>

<details>
<summary>Solution</summary>

```python
week = pd.DataFrame(requests.get(URL, params={"providers": "ecb", "from": "2026-02-02", "to": "2026-02-06", "base": "USD", "quotes": "EUR,GBP"}, timeout=30).json())
euro = week[week["quote"] == "EUR"]
answer_1 = euro["rate"].mean()
```

The answer is 0.8467: the five euro rates of the week are 0.84459, 0.84739, 0.84602, 0.84760 and 0.84789. Averaging all ten rows gives 0.7897, because the pound rows, around 0.73, are in it; the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many orders did the customers of one city place? Before you start, say in one sentence what the function gets and gives back. Write the function `orders_in_city(city)` in the empty cell of the notebook: it asks the database `con` of unit 3 for the number of orders whose customer lives in `city` (a join of `orders` and `customers`) and returns that number. The city arrives as text from outside, so it must not be pasted into the query.

**Expected:** a function `orders_in_city` that returns an integer.

In [ ]:
_ = check_function("Exercise 2", "orders_in_city", [(("Oakland",), {}, 689), (("Berkeley",), {}, 706), (("Oakland' OR '1'='1",), {}, 0)])

<details>
<summary>Hint</summary>

The join is `orders JOIN customers ON orders.customer_id = customers.customer_id`, and the city goes in as `?`. Check the function with a city that is not one.

</details>

<details>
<summary>Solution</summary>

```python
def orders_in_city(city):
    sql = "SELECT COUNT(*) AS n FROM orders JOIN customers ON orders.customer_id = customers.customer_id WHERE customers.city = ?"
    return int(pd.read_sql(sql, con, params=(city,)).iloc[0, 0])
```

The counts are 689 for Oakland, 706 for Berkeley and 404 for Alameda; the three cities add up to 1,799, because the walk-in order has no customer. A function that pastes the city into the query gives the same numbers for the honest names and 1,799 for `Oakland' OR '1'='1`, which is all the orders that have a customer; the check cell tries that text.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A supplier in Frankfurt sends an invoice of 1,000 euros, and the owner wants to know what it cost in dollars on January 5, 2026. In these answers the rate is euros for one dollar, as in unit 2. The colleague's answer is 857.34. Fix the line, then set `answer_3` to the cost in dollars, a decimal number.

**Expected:** one decimal number, above 1,000.

In [ ]:
day = pd.DataFrame(requests.get(URL, params={"providers": "ecb", "date": "2026-01-05", "base": "USD", "quotes": "EUR"}, timeout=30).json())
invoice_usd = 1000 * day["rate"].iloc[0]  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = invoice_usd

In [ ]:
_ = check("Exercise 3", "answer_3", 1000 / 0.85734, traps={857.34: "That multiplies euros by the euros that one dollar buys. One dollar buys less than one euro here, so 1,000 euros must cost more than 1,000 dollars."})

<details>
<summary>Hint</summary>

In unit 2 a dollar amount times the rate gave euros. Which operation turns euros back into dollars?

</details>

<details>
<summary>Solution</summary>

```python
day = pd.DataFrame(requests.get(URL, params={"providers": "ecb", "date": "2026-01-05", "base": "USD", "quotes": "EUR"}, timeout=30).json())
invoice_usd = 1000 / day["rate"].iloc[0]
answer_3 = invoice_usd
```

The answer is 1,166.40 dollars: 1,000 euros divided by 0.85734 euros for each dollar. Multiplying gives 857.34, a number below 1,000, so it cannot be the price of 1,000 euros in a currency that is worth less; the check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")